<a href="https://colab.research.google.com/github/saverin0/Change-Detection-Using-Dinov3/blob/main/01_config_and_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpaceNet-7 Temporal Change Detection with DINOv3
## Part 1: Configuration and Label Extraction

### What this notebook does
SpaceNet-7 has 60 training locations (AOIs). Each one has about 2 years of **monthly** satellite images and a **monthly GeoJSON** of building footprints. Buildings keep the same `Id` from month to month, so we can follow each building over time.

This notebook turns those raw footprints into **training labels** for change detection. It creates two kinds of labels:

| Label | Granularity | Question it answers |
|---|---|---|
| `pixel_change_masks` | per pixel, per month pair | *Did this pixel go from building to no building (or back) between month t and t+1?* |
| `building_labels` | per building, per month | *What happened to this building: stable, new, demolished, expanded, reduced, or modified?* |

Part 2 extracts DINOv3 image features. Part 3 trains a temporal model on those features, using the labels made here as targets.

### Pipeline at a glance
```
Mount Drive ─► Install deps ─► Define Config + label functions ─► Build Config
     ─► Check data & resources ─► Stage Drive files to local SSD
     ─► Extract labels (parallel) ─► Sync labels to Drive ─► Summarize ─► Verify .npz cache
     ─► Build cloud-mask maps ─► Create train / val / test split
```

The notebook is **self-contained**: all the code it needs is defined in its own cells, with no repo to clone.

> **Cloud masks:** SpaceNet-7's `images_masked/` blacks out cloudy parts of each month, and buildings under those areas are left out of that month's labels. Buildings therefore "disappear" and "reappear" wherever clouds were, and **88% of the raw change-label pixels lie inside masked areas**, which cover only ~6% of the image. Step 13 records which pixels are masked in each month, so Part 3 can ignore them instead of learning to detect clouds.

### Change types
| Index | Type | Rule, comparing a building with its previous month |
|---|---|---|
| 0 | `STABLE` | Same footprint (shape IoU ≥ 0.80, area change within ±10%) |
| 1 | `NEW` | `Id` shows up for the first time (or its previous area was 0) |
| 2 | `DEMOLISHED` | `Id` was there last month and is gone now |
| 3 | `EXPANDED` | Area grew by ≥ 10% |
| 4 | `REDUCED` | Area shrank by ≥ 10% |
| 5 | `MODIFIED` | Area about the same, but shape IoU < 0.80 (reshaped or moved) |

## Step 1: Mount Google Drive

The SpaceNet-7 dataset and all outputs (config, label cache, and later features and model checkpoints) live on Google Drive. That way they survive after the Colab runtime is recycled. Mounting Drive makes it show up as a normal folder at `/content/drive/MyDrive/...`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Install dependencies and import libraries

Colab already includes almost everything this notebook uses: NumPy, GeoPandas, Shapely 2, joblib, tqdm, and psutil. The one exception is **rasterio**, which reads GeoTIFFs and draws footprints onto the pixel grid. This cell installs it only if it's missing.

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rasterio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])

import json
import multiprocessing as mp
import shutil
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from zoneinfo import ZoneInfo

import geopandas as gpd
import numpy as np
import psutil
import rasterio
from joblib import Parallel, delayed
from rasterio.features import rasterize
from shapely import area, intersection, is_empty, is_valid, union
from tqdm.auto import tqdm

print(f"CPU cores available: {mp.cpu_count()}")

CPU cores available: 2


## Step 3: Define the `Config` class

`Config` is a dataclass that holds every path, threshold, and hyperparameter for the whole project, so all settings live in one place. This notebook uses the paths, image settings, change thresholds, and `n_jobs`. The training and model settings are here so they get written to `config.json` for Parts 2 and 3.

Helpers on the class:
- `cache_dir`: equals `cache_base`, unless `use_timestamped_cache=True` gives each run its own versioned folder.
- `patches_per_side`: `image_size // patch_size`, i.e. 512 / 16 = **32** DINOv3 patches per side.
- `make_cache_dirs()`: creates `cache_dir/`, `labels/`, and `features/`.
- `save()` / `load_dict()`: write and read `cache_dir/config.json`. That file is how later notebooks pick up the same settings.

In [3]:
@dataclass
class Config:
    # Data paths -- update these for your Drive layout.
    data_root: str = "/content/drive/MyDrive/datasets/spacenet7/SN7_buildings_train/train"
    cache_base: str = "/content/drive/MyDrive/datasets/spacenet7_cache"

    # Set True to get a fresh timestamped cache_dir instead of reusing cache_base directly.
    use_timestamped_cache: bool = False
    timestamp: str = field(
        default_factory=lambda: datetime.now(ZoneInfo("Europe/Berlin")).strftime("%Y%m%d_%H%M%S")
    )

    # Image settings (DINOv3 patch size 16).
    image_size: int = 512
    patch_size: int = 16

    # DINOv3 SAT-493M (satellite-pretrained).
    dino_model: str = "facebook/dinov3-vitl16-pretrain-sat493m"
    hidden_dim: int = 1024

    # Change detection thresholds.
    expand_threshold: float = 0.10
    reduce_threshold: float = 0.10
    shape_iou_threshold: float = 0.80

    # Training settings.
    train_split: float = 0.80
    batch_size: int = 4
    epochs: int = 50
    learning_rate: float = 1e-4
    weight_decay: float = 0.01

    # Model architecture.
    transformer_dim: int = 256
    transformer_heads: int = 8
    transformer_layers: int = 4
    dropout: float = 0.1
    max_months: int = 26
    output_resolution: int = 128

    # Parallelization for label extraction.
    n_jobs: int = -1

    change_types: List[str] = field(
        default_factory=lambda: ["STABLE", "NEW", "DEMOLISHED", "EXPANDED", "REDUCED", "MODIFIED"]
    )

    @property
    def cache_dir(self) -> str:
        if self.use_timestamped_cache:
            return f"{self.cache_base}_{self.timestamp}"
        return self.cache_base

    @property
    def num_change_types(self) -> int:
        return len(self.change_types)

    @property
    def patches_per_side(self) -> int:
        return self.image_size // self.patch_size

    def to_dict(self) -> dict:
        return {
            "data_root": self.data_root,
            "cache_dir": self.cache_dir,
            "image_size": self.image_size,
            "patch_size": self.patch_size,
            "hidden_dim": self.hidden_dim,
            "dino_model": self.dino_model,
            "change_types": self.change_types,
            "expand_threshold": self.expand_threshold,
            "reduce_threshold": self.reduce_threshold,
            "max_months": self.max_months,
            "output_resolution": self.output_resolution,
            "epochs": self.epochs,
            "timestamp": self.timestamp,
        }

    def make_cache_dirs(self) -> None:
        Path(self.cache_dir).mkdir(parents=True, exist_ok=True)
        (Path(self.cache_dir) / "features").mkdir(exist_ok=True)
        (Path(self.cache_dir) / "labels").mkdir(exist_ok=True)

    def save(self) -> Path:
        path = Path(self.cache_dir) / "config.json"
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)
        return path

    @staticmethod
    def load_dict(cache_dir: str) -> dict:
        with open(Path(cache_dir) / "config.json") as f:
            return json.load(f)

## Step 4: Define the label-extraction functions

These functions do the actual work. Step 9 runs them on all 60 locations in parallel.

| Function | Role |
|---|---|
| `get_month_from_filename` | Gets `YYYY_MM` from names like `global_monthly_2018_01_...` |
| `load_buildings_geopandas` | Loads one month's GeoJSON and keeps only valid single `Polygon`s |
| `compute_changes_vectorized` | Classifies every building between two months |
| `process_location_labels_optimized` | Builds all masks and labels for one location |
| `process_single_location` | Cache-aware wrapper: skip if already done, otherwise compute and save `.npz` |

### What happens for each location
1. **Cache check:** if `labels/<loc_id>_labels.npz` already exists, skip the location and only read its summary counts. Re-running the notebook is cheap.
2. **Georeferencing:** open the first monthly GeoTIFF to get its affine `transform` and pixel `shape`. This lets us draw footprints (map coordinates) onto the image's pixel grid.
3. **Load footprints per month:** read each monthly GeoJSON with GeoPandas.
   - Drop invalid and empty geometries.
   - For a `MultiPolygon`, keep only its largest part, so every building is one `Polygon`.
4. **Building masks:** rasterize each month's footprints into a binary mask. Result: `building_masks` with shape `(T, H, W)`.
5. **Pixel change masks:** compare consecutive months pixel by pixel (`mask[t] != mask[t+1]`). Result: `pixel_change_masks` with shape `(T-1, H, W)`.
6. **Per-building tracking:**
   - Collect every building `Id` seen in any month.
   - Build an `existence` matrix of shape `(N_buildings, T)`.
7. **Classify changes between consecutive months** with vectorized Shapely 2.0 operations (`area`, `intersection`, `union`), using the rules from the table at the top:
   - Ids that appear → `NEW`; Ids that disappear → `DEMOLISHED`.
   - Ids present in both months are compared by **area ratio** and **shape IoU** → `EXPANDED`, `REDUCED`, `MODIFIED`, or `STABLE`.
   - Months with no footprints at all are skipped. The comparison goes back to the last non-empty month, so one missing month doesn't erase every later change.
8. **Summary label:** each building gets `summary_types`, its **first** non-stable change, plus the month it happened (`summary_months`).
9. **Save:** write everything to a compressed `.npz` in the labels cache.

In [4]:
def get_month_from_filename(filename: str) -> Optional[str]:
    """Extract YYYY_MM from filenames like 'global_monthly_2018_01_...'."""
    parts = Path(filename).stem.split("_")
    if len(parts) >= 4 and parts[0] == "global" and parts[1] == "monthly":
        return f"{parts[2]}_{parts[3]}"
    return None


def load_buildings_geopandas(geojson_path: Path) -> gpd.GeoDataFrame:
    """Load buildings via GeoPandas, keeping only valid single Polygons."""
    try:
        gdf = gpd.read_file(geojson_path)
        if gdf.empty:
            return gpd.GeoDataFrame(columns=["Id", "geometry"])

        if "Id" not in gdf.columns:
            gdf["Id"] = range(len(gdf))

        valid_mask = is_valid(gdf.geometry.values) & ~is_empty(gdf.geometry.values)
        gdf = gdf[valid_mask].copy()

        def to_polygon(geom):
            if geom.geom_type == "MultiPolygon":
                return max(geom.geoms, key=lambda g: g.area)
            return geom

        gdf["geometry"] = gdf.geometry.apply(to_polygon)
        gdf = gdf[gdf.geometry.geom_type == "Polygon"]

        return gdf[["Id", "geometry"]]
    except Exception:
        return gpd.GeoDataFrame(columns=["Id", "geometry"])


def compute_changes_vectorized(
    prev_gdf: gpd.GeoDataFrame,
    curr_gdf: gpd.GeoDataFrame,
    all_ids: np.ndarray,
    config: Config,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute per-building change type/confidence between two months (Shapely 2.0 vectorized)."""
    n_buildings = len(all_ids)
    change_types = np.zeros(n_buildings, dtype=np.int64)
    confidences = np.zeros(n_buildings, dtype=np.float32)

    prev_dict = dict(zip(prev_gdf["Id"].values, prev_gdf.geometry.values)) if not prev_gdf.empty else {}
    curr_dict = dict(zip(curr_gdf["Id"].values, curr_gdf.geometry.values)) if not curr_gdf.empty else {}

    prev_ids = set(prev_dict.keys())
    curr_ids = set(curr_dict.keys())

    new_ids = curr_ids - prev_ids
    demolished_ids = prev_ids - curr_ids
    common_ids = prev_ids & curr_ids

    id_to_idx = {bid: idx for idx, bid in enumerate(all_ids)}

    for bid in new_ids:
        if bid in id_to_idx:
            idx = id_to_idx[bid]
            change_types[idx] = 1  # NEW
            confidences[idx] = 1.0

    for bid in demolished_ids:
        if bid in id_to_idx:
            idx = id_to_idx[bid]
            change_types[idx] = 2  # DEMOLISHED
            confidences[idx] = 1.0

    if common_ids:
        common_list = list(common_ids)
        prev_geoms = np.array([prev_dict[bid] for bid in common_list])
        curr_geoms = np.array([curr_dict[bid] for bid in common_list])

        prev_areas = area(prev_geoms)
        curr_areas = area(curr_geoms)

        intersection_areas = area(intersection(prev_geoms, curr_geoms))
        union_areas = area(union(prev_geoms, curr_geoms))

        with np.errstate(divide="ignore", invalid="ignore"):
            area_ratios = np.where(prev_areas > 0, (curr_areas - prev_areas) / prev_areas, 0)
            shape_ious = np.where(union_areas > 0, intersection_areas / union_areas, 0)

        for i, bid in enumerate(common_list):
            if bid not in id_to_idx:
                continue
            idx = id_to_idx[bid]

            ar = area_ratios[i]
            iou = shape_ious[i]

            if prev_areas[i] == 0:
                change_types[idx] = 1  # NEW
                confidences[idx] = 1.0
            elif ar >= config.expand_threshold:
                change_types[idx] = 3  # EXPANDED
                confidences[idx] = min(1.0, ar / 0.5)
            elif ar <= -config.reduce_threshold:
                change_types[idx] = 4  # REDUCED
                confidences[idx] = min(1.0, abs(ar) / 0.5)
            elif iou < config.shape_iou_threshold:
                change_types[idx] = 5  # MODIFIED
                confidences[idx] = 1.0 - iou
            else:
                change_types[idx] = 0  # STABLE
                confidences[idx] = iou

    return change_types, confidences


def process_location_labels_optimized(loc_path: Path, config: Config) -> Dict:
    """Extract pixel/building-level change labels for one location."""
    labels_path = loc_path / "labels_match"
    images_path = loc_path / "images_masked"

    if not labels_path.exists():
        raise ValueError(f"Labels not found: {labels_path}")

    image_files = sorted(images_path.glob("*.tif"))
    if not image_files:
        image_files = sorted((loc_path / "images").glob("*.tif"))

    with rasterio.open(image_files[0]) as src:
        image_transform = src.transform
        image_shape = (src.height, src.width)

    label_files = sorted(labels_path.glob("*.geojson"))
    months = []
    monthly_gdfs = {}
    all_building_ids = set()

    for label_file in label_files:
        month = get_month_from_filename(label_file.name)
        if month:
            months.append(month)
            gdf = load_buildings_geopandas(label_file)
            monthly_gdfs[month] = gdf
            if not gdf.empty:
                all_building_ids.update(gdf["Id"].values)

    months = sorted(set(months))
    n_months = len(months)

    building_masks = np.zeros((n_months, *image_shape), dtype=np.uint8)
    for t, month in enumerate(months):
        gdf = monthly_gdfs.get(month)
        if gdf is not None and not gdf.empty:
            shapes = [(geom, 1) for geom in gdf.geometry.values]
            building_masks[t] = rasterize(
                shapes, out_shape=image_shape, transform=image_transform, fill=0, dtype=np.uint8
            )

    pixel_change_masks = (building_masks[:-1] != building_masks[1:]).astype(np.uint8)

    building_ids = np.array(sorted(all_building_ids))
    n_buildings = len(building_ids)

    if n_buildings == 0:
        return {
            "pixel_change_masks": pixel_change_masks,
            "building_masks": building_masks,
            "building_labels": {
                "building_ids": [],
                "existence": np.zeros((0, n_months), dtype=bool),
                "change_types": np.zeros((0, n_months), dtype=np.int64),
                "summary_types": np.zeros(0, dtype=np.int64),
                "summary_months": np.full(0, -1, dtype=np.int64),
                "confidences": np.zeros((0, n_months), dtype=np.float32),
            },
            "months": months,
            "image_shape": image_shape,
        }

    existence = np.zeros((n_buildings, n_months), dtype=bool)
    change_types = np.zeros((n_buildings, n_months), dtype=np.int64)
    confidences = np.zeros((n_buildings, n_months), dtype=np.float32)

    id_to_idx = {bid: idx for idx, bid in enumerate(building_ids)}
    for t, month in enumerate(months):
        gdf = monthly_gdfs.get(month)
        if gdf is not None and not gdf.empty:
            for bid in gdf["Id"].values:
                if bid in id_to_idx:
                    existence[id_to_idx[bid], t] = True

    # Walk back past months with no footprints so one missing month doesn't
    # blank out every change after it.
    for t in range(1, n_months):
        curr_gdf = monthly_gdfs.get(months[t], gpd.GeoDataFrame())
        if curr_gdf.empty:
            continue

        prev_gdf = None
        prev_t = t - 1
        while prev_t >= 0:
            prev_gdf = monthly_gdfs.get(months[prev_t], gpd.GeoDataFrame())
            if not prev_gdf.empty:
                break
            prev_t -= 1

        if prev_gdf is None or prev_gdf.empty:
            continue

        ct, conf = compute_changes_vectorized(prev_gdf, curr_gdf, building_ids, config)
        change_types[:, t] = ct
        confidences[:, t] = conf

    summary_types = np.zeros(n_buildings, dtype=np.int64)
    summary_months = np.full(n_buildings, -1, dtype=np.int64)

    non_stable_mask = change_types != 0
    for i in range(n_buildings):
        non_stable_indices = np.where(non_stable_mask[i])[0]
        if len(non_stable_indices) > 0:
            first_change_t = non_stable_indices[0]
            summary_types[i] = change_types[i, first_change_t]
            summary_months[i] = first_change_t

    return {
        "pixel_change_masks": pixel_change_masks,
        "building_masks": building_masks,
        "building_labels": {
            "building_ids": building_ids.tolist(),
            "existence": existence,
            "change_types": change_types,
            "summary_types": summary_types,
            "summary_months": summary_months,
            "confidences": confidences,
        },
        "months": months,
        "image_shape": image_shape,
    }


def process_single_location(loc_path: Path, config: Config, cache_dir: Path) -> Dict:
    """Process one location and cache to disk; returns summary stats for aggregation."""
    loc_id = loc_path.name
    cache_path = cache_dir / "labels" / f"{loc_id}_labels.npz"

    result = {
        "loc_id": loc_id,
        "status": "unknown",
        "n_buildings": 0,
        "change_counts": defaultdict(int),
        "error": None,
    }

    if cache_path.exists():
        try:
            data = np.load(cache_path, allow_pickle=True)
            building_labels = data["building_labels"].item()
            result["status"] = "cached"
            result["n_buildings"] = len(building_labels["summary_types"])
            for idx in building_labels["summary_types"]:
                result["change_counts"][idx] += 1
        except Exception:
            result["status"] = "cache_error"
        return result

    try:
        labels = process_location_labels_optimized(loc_path, config)
        np.savez_compressed(cache_path, **labels)

        result["status"] = "processed"
        result["n_buildings"] = len(labels["building_labels"]["summary_types"])
        for idx in labels["building_labels"]["summary_types"]:
            result["change_counts"][idx] += 1

    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)

    return result

## Step 5: Build and save the configuration

This step creates a `Config` with your Drive paths and saves it. The settings that matter most here:

| Setting | Value | Meaning |
|---|---|---|
| `data_root` | `.../SN7_buildings_train/train` | Folder with the 60 `L15-*` location folders |
| `cache_base` / `cache_dir` | `.../spacenet7_cache` | Where labels, features, and models are written |
| `image_size` / `patch_size` | 512 / 16 | Images are resized to 512 px for DINOv3, giving a **32×32** patch grid |
| `expand_threshold` / `reduce_threshold` | 0.10 | ±10% area change counts as EXPANDED / REDUCED |
| `shape_iou_threshold` | 0.80 | IoU below this (with similar area) counts as MODIFIED |
| `n_jobs` | -1 | Use every CPU core for parallel label extraction |

`make_cache_dirs()` creates the cache folders on Drive. `save()` writes `config.json`, which **later notebooks read** to get the same paths and settings.

In [5]:
# CONFIGURATION -- update data_root / cache_base for your Drive layout.
config = Config(
    data_root="/content/drive/MyDrive/datasets/spacenet7/SN7_buildings_train/train",
    cache_base="/content/drive/MyDrive/datasets/spacenet7_cache",
)
# config.cache_dir == config.cache_base by default (no auto timestamp), so Parts 2 and 3
# find this run automatically as long as they use the same cache_base -- no manual
# reconciliation needed. Set use_timestamped_cache=True above for a fresh versioned run dir.

config.make_cache_dirs()
config.save()

print("Configuration:")
print(f"   Data root: {config.data_root}")
print(f"   Cache dir: {config.cache_dir}")
print(f"   Image size: {config.image_size} (patches: {config.patches_per_side}x{config.patches_per_side})")
print(f"   Parallel jobs: {config.n_jobs}")

Configuration:
   Data root: /content/drive/MyDrive/datasets/spacenet7/SN7_buildings_train/train
   Cache dir: /content/drive/MyDrive/datasets/spacenet7_cache
   Image size: 512 (patches: 32x32)
   Parallel jobs: -1


## Step 6: Verify the dataset is where we expect

A quick check before the heavy work: does `data_root` exist, and how many location folders does it hold? Each location folder is named after its map tile (e.g. `L15-0331E-1257N_1327_3160_13`) and looks like this:

```
L15-.../
├── images/ or images_masked/   # one GeoTIFF per month (global_monthly_YYYY_MM_...tif)
└── labels_match/               # one GeoJSON per month; building Ids match across months
```

You should see **60 locations**. If the path is wrong, fix `data_root` in Step 5.

In [6]:
# Verify data path exists
root = Path(config.data_root)
if not root.exists():
    print(f"Data path does not exist: {config.data_root}")
    print("Update config.data_root above")
else:
    locations = sorted([d for d in root.iterdir() if d.is_dir() and d.name.startswith("L15-")])
    print(f"Found {len(locations)} locations")
    print(f"   Sample: {locations[0].name}")

Found 60 locations
   Sample: L15-0331E-1257N_1327_3160_13


## Step 7: Check available compute

Label extraction runs one worker process per CPU core, and each worker holds a full stack of monthly masks in memory. This cell shows how many cores and how much RAM are available.

A free or standard Colab runtime has **2 cores and ~13 GB RAM**, so two locations are processed at a time. A High-RAM runtime (8 cores) is roughly 4× faster for Step 9.

In [7]:
# Resource check
cpu_count = mp.cpu_count()
mem = psutil.virtual_memory()
print(f"CPU cores available: {cpu_count}")
print(f"Memory: {mem.available/1e9:.1f} GB available / {mem.total/1e9:.1f} GB total")

CPU cores available: 2
Memory: 12.6 GB available / 13.6 GB total


## Step 8: Stage the data from Drive onto Colab's local disk

Google Drive is a network drive. Every file open, check, or read goes over the network and takes about 100–300 ms, no matter how small the file is. Label extraction reads ~24 GeoJSONs for each of the 60 locations (~1,500 files). Reading them straight from Drive would mean the workers spend much of their time waiting on the network.

This cell copies what extraction needs onto Colab's local SSD (`/content/sn7_part1_*`):
- every monthly GeoJSON in `labels_match/`
- **only the first** GeoTIFF of each location, since extraction only reads it for georeferencing

It copies with **32 threads at once**. The work is mostly waiting on the network, not CPU, so many threads speed it up even on a 2-core runtime.

**Locations that already have labels on Drive are not re-copied.** Only their cached `.npz` is copied, so Step 9 can count them without redoing any work. Once all 60 are done, a re-run copies 60 small files instead of ~1,500. Files already on local disk with the same size are skipped too.

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _copy_if_stale(source_file: Path, target_file: Path) -> None:
    target_file.parent.mkdir(parents=True, exist_ok=True)
    if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
        shutil.copy2(source_file, target_file)


def run_copies(copy_jobs, desc):
    with ThreadPoolExecutor(max_workers=32) as pool:
        futures = [pool.submit(_copy_if_stale, src, dst) for src, dst in copy_jobs]
        for future in tqdm(as_completed(futures), total=len(futures), desc=desc):
            future.result()


drive_root = Path(config.data_root)
drive_labels = Path(config.cache_dir) / "labels"
processing_root = Path("/content/sn7_part1_data")
processing_cache = Path("/content/sn7_part1_cache")
(processing_cache / "labels").mkdir(parents=True, exist_ok=True)

drive_locations = sorted(d for d in drive_root.iterdir() if d.is_dir() and d.name.startswith("L15-"))
cached_ids = {f.name.removesuffix("_labels.npz") for f in drive_labels.glob("*_labels.npz")}

copy_jobs = []
for source_location in drive_locations:
    if source_location.name in cached_ids:
        continue  # labels already on Drive; Step 9 only needs the cached .npz
    local_location = processing_root / source_location.name

    for label_file in (source_location / "labels_match").glob("*.geojson"):
        copy_jobs.append((label_file, local_location / "labels_match" / label_file.name))

    source_images = source_location / "images_masked"
    if not source_images.exists():
        source_images = source_location / "images"
    first_image = min(source_images.glob("*.tif"), default=None)
    if first_image is None:
        raise FileNotFoundError(f"No GeoTIFF found for {source_location.name}")
    copy_jobs.append((first_image, local_location / source_images.name / first_image.name))

for cached_file in drive_labels.glob("*.npz"):
    copy_jobs.append((cached_file, processing_cache / "labels" / cached_file.name))

run_copies(copy_jobs, "Staging from Drive")
print(f"To process: {len(drive_locations) - len(cached_ids)} | Already cached: {len(cached_ids)}")
print(f"Staged {len(copy_jobs)} files to local disk")

Staging from Drive:   0%|          | 0/60 [00:00<?, ?it/s]

To process: 0 | Already cached: 60
Staged 60 files to local disk


## Step 9: Extract change labels for every location (parallel)

This is the main step. `joblib.Parallel` with the `loky` backend runs `process_single_location` (from Step 4) in separate processes, one location per worker. Workers read the **local staged copy** from Step 8 and write `.npz` files to local disk. Step 10 copies them to Drive.

The progress bar counts **finished** locations. It uses `return_as="generator_unordered"`, which hands back each result as soon as its worker finishes.

When all workers finish, the cell adds up the per-location results:
- number of locations processed, loaded from cache, or failed
- total building count
- a count for each change type (from each building's summary label)

A cached `.npz` that can't be read (`cache_error`, e.g. a file cut short by a disconnect) is **counted as an error**. To rebuild that location, delete its file from Drive and run again.

In [9]:
# Process all locations (parallel) from the staged local copy
# Cached locations have no staged folder; process_single_location finds their .npz
# in the local cache first and never opens the folder.
locations = [processing_root / d.name for d in drive_locations]
cache_dir = processing_cache

print(f"Found {len(locations)} locations")
print(f"Using {mp.cpu_count() if config.n_jobs == -1 else config.n_jobs} parallel workers")

# return_as="generator_unordered" yields each result as soon as its worker finishes,
# so the progress bar tracks completed locations. Wrapping the *input* in tqdm (as
# before) only tracks dispatch, which joblib does all at once, so the bar sat at 0.
results_iter = Parallel(n_jobs=config.n_jobs, backend="loky", return_as="generator_unordered")(
    delayed(process_single_location)(loc_path, config, cache_dir)
    for loc_path in locations
)
results = list(tqdm(results_iter, total=len(locations), desc="Processing labels"))

change_distribution = defaultdict(int)
total_buildings = 0
processed = 0
cached = 0
errors = []

for res in results:
    total_buildings += res["n_buildings"]
    for idx, count in res["change_counts"].items():
        change_distribution[config.change_types[idx]] += count
    if res["status"] == "processed":
        processed += 1
    elif res["status"] == "cached":
        cached += 1
    elif res["status"] in ("error", "cache_error"):
        errors.append({"loc_id": res["loc_id"], "error": res["error"] or res["status"]})

print(f"Processed: {processed} | Cached: {cached} | Errors: {len(errors)}")
if errors:
    for err in errors[:5]:
        print(f"   {err['loc_id']}: {err['error']}")

Found 60 locations
Using 2 parallel workers


Processing labels:   0%|          | 0/60 [00:00<?, ?it/s]

Processed: 0 | Cached: 60 | Errors: 0


## Step 10: Sync label files back to Drive

Colab's local disk is erased when the runtime shuts down, so the `.npz` files are copied back to `cache_dir/labels/` on Drive. Later notebooks read them from there. Files already on Drive with the same size are skipped, so only newly processed locations are uploaded.

In [10]:
# Sync newly written label files from local disk back to Drive
local_label_files = sorted((processing_cache / "labels").glob("*.npz"))
drive_labels.mkdir(parents=True, exist_ok=True)
run_copies([(f, drive_labels / f.name) for f in local_label_files], "Syncing to Drive")
print(f"Synced {len(local_label_files)} label files to {drive_labels}")

Syncing to Drive:   0%|          | 0/60 [00:00<?, ?it/s]

Synced 60 label files to /content/drive/MyDrive/datasets/spacenet7_cache/labels


## Step 11: Change-type distribution

This prints how many buildings fall into each change type. Each building is counted **once**, by its summary label (its first non-stable change, or `STABLE` if it never changed).

### Reading the numbers
- **The classes are very imbalanced.** `STABLE`, `NEW`, and `DEMOLISHED` make up over 99% of buildings. `EXPANDED`, `REDUCED`, and `MODIFIED` are well under 1%. Training will need a loss that handles imbalance (e.g. Focal + Dice) and mask downsampling that keeps rare positives (max-pooling), or the model could ignore the rare "change" pixels.
- **`DEMOLISHED` and `NEW` are heavily inflated by cloud masks.** A building under a cloud-masked area is left out of that month's labels, so it counts as `DEMOLISHED` (and `NEW` when it reappears), even though nothing was built or torn down. At pixel level, **88% of change-label pixels are inside masked areas**. These per-building counts describe the raw labels, not real construction. Part 3 uses the mask maps from Step 13 to ignore masked pixels during training and evaluation.

In [11]:
total = sum(change_distribution.values())
print("Change type distribution:")
for ct in config.change_types:
    count = change_distribution[ct]
    pct = 100 * count / total if total > 0 else 0
    print(f"   {ct:12s}: {count:6,d} ({pct:5.1f}%)")

print(f"\nLabels saved to: {config.cache_dir}/labels/")

Change type distribution:
   STABLE      : 183,571 ( 57.5%)
   NEW         : 36,198 ( 11.3%)
   DEMOLISHED  : 98,014 ( 30.7%)
   EXPANDED    :    321 (  0.1%)
   REDUCED     :  1,281 (  0.4%)
   MODIFIED    :     19 (  0.0%)

Labels saved to: /content/drive/MyDrive/datasets/spacenet7_cache/labels/


## Step 12: Verify the label cache

A final check: count the `.npz` files on Drive (should be 60) and open one to confirm what's inside.

| Key | Shape (example) | Contents |
|---|---|---|
| `building_masks` | `(25, 1023, 1024)` | Binary building footprint for each of the T = 25 months |
| `pixel_change_masks` | `(24, 1023, 1024)` | Binary change between each consecutive month pair (T-1 = 24) |
| `building_labels` | dict | `building_ids`, `existence (N,T)`, `change_types (N,T)`, `confidences (N,T)`, `summary_types (N,)`, `summary_months (N,)` |
| `months` | list | `YYYY_MM` strings, in time order |
| `image_shape` | tuple | `(H, W)` of the original GeoTIFF |

The masks are at the **original image resolution** (~1024 px). Training will downsample them (e.g. to 128 px) with max-pooling.

In [12]:
# Verify cache
cache_files = list((Path(config.cache_dir) / "labels").glob("*.npz"))
print(f"Cached label files: {len(cache_files)}")

if cache_files:
    sample = np.load(cache_files[0], allow_pickle=True)
    print(f"Sample: {cache_files[0].name}")
    print(f"   pixel_change_masks: {sample['pixel_change_masks'].shape}")
    print(f"   building_masks: {sample['building_masks'].shape}")
    building_labels = sample["building_labels"].item()
    print(f"   buildings: {len(building_labels['building_ids'])}")

print("\nLabel cache verified.")

Cached label files: 60
Sample: L15-0331E-1257N_1327_3160_13_labels.npz
   pixel_change_masks: (24, 1023, 1024)
   building_masks: (25, 1023, 1024)
   buildings: 2156

Label cache verified.


## Step 13: Build cloud-mask maps

`images_masked/` blacks out cloud-covered parts of each monthly image, and SpaceNet leaves the buildings under those areas out of that month's labels. That creates fake change: a building "disappears" when a cloud covers it and "reappears" afterwards. Measured over all 60 locations:

| | Share |
|---|---|
| Image pixels masked in either month of a pair | ~6% |
| **Change-label pixels inside those masked areas** | **~88%** |

A model trained on the raw labels mostly learns to find the black masked areas, which is easy and inflates the scores. This step records **where each month is masked**, so Part 3 can ignore those pixels in the loss and the metrics.

For each location and each month in its label file:
1. **Read** the monthly GeoTIFF from `images_masked/` on Drive.
2. **Find masked pixels:** all three RGB bands are 0, or the alpha band is 0.
3. **Shrink to 128×128 with max-pooling**, the same grid Part 3 uses for its targets. An output cell counts as masked if *any* pixel inside it is masked, so ignored areas fully cover the masked region, including its edges.
4. **Save** `masks/<loc_id>_masks.npz` with `masked` `(T, 128, 128)` bool and `months`, in the same order as the labels.

The ~1,400 images are read straight from Drive with 12 threads; each thread handles one location. Locations whose mask file already exists with matching months are skipped, so re-running is cheap. Part 2's features are not affected and do **not** need to be re-extracted.

In [13]:
import torch
import torch.nn.functional as F

MASK_RES = config.output_resolution
masks_dir = Path(config.cache_dir) / "masks"
masks_dir.mkdir(parents=True, exist_ok=True)


def month_image(location_dir, month):
    folder = location_dir / "images_masked"
    matches = sorted(folder.glob(f"global_monthly_{month}_*.tif"))
    if not matches:
        raise FileNotFoundError(f"No image in {folder} for month {month}")
    return matches[0]


def build_location_masks(label_file):
    loc_id = label_file.name.removesuffix("_labels.npz")
    out_path = masks_dir / f"{loc_id}_masks.npz"
    with np.load(label_file, allow_pickle=True) as labels:
        months = [str(m) for m in labels["months"]]

    if out_path.exists():
        with np.load(out_path) as existing:
            if [str(m) for m in existing["months"]] == months:
                return loc_id, "cached", None

    try:
        masked = []
        for month in months:
            with rasterio.open(month_image(Path(config.data_root) / loc_id, month)) as src:
                img = src.read()
            pixel_masked = img[:3].max(axis=0) == 0
            if img.shape[0] >= 4:
                pixel_masked |= img[3] == 0
            masked.append(torch.from_numpy(pixel_masked))
        stacked = torch.stack(masked).float().unsqueeze(1)
        pooled = F.adaptive_max_pool2d(stacked, MASK_RES).squeeze(1).bool().numpy()
        np.savez_compressed(out_path, masked=pooled, months=np.array(months))
        return loc_id, "processed", None
    except Exception as error:
        return loc_id, "error", f"{type(error).__name__}: {error}"


label_files = sorted(drive_labels.glob("*_labels.npz"))
with ThreadPoolExecutor(max_workers=12) as pool:
    mask_results = list(tqdm(pool.map(build_location_masks, label_files), total=len(label_files), desc="Cloud masks"))

mask_status = defaultdict(int)
for _, status, _ in mask_results:
    mask_status[status] += 1
print(f"Mask maps: processed {mask_status['processed']} | cached {mask_status['cached']} | errors {mask_status['error']}")
for loc_id, status, message in mask_results:
    if status == "error":
        print(f"   {loc_id}: {message}")

mask_files = sorted(masks_dir.glob("*_masks.npz"))
masked_share = []
for mask_file in mask_files:
    with np.load(mask_file) as m:
        pair_masked = m["masked"][:-1] | m["masked"][1:]
        masked_share.append(pair_masked.mean())
print(f"Mask files: {len(mask_files)} / {len(label_files)} locations -> {masks_dir}")
if masked_share:
    print(f"   Average share of 128x128 cells masked in a month pair: {100 * np.mean(masked_share):.1f}%")

if mask_status["error"] == 0 and len(mask_files) == len(label_files):
    print("\nCloud masks complete.")

Cloud masks:   0%|          | 0/60 [00:00<?, ?it/s]

Mask maps: processed 60 | cached 0 | errors 0
Mask files: 60 / 60 locations -> /content/drive/MyDrive/datasets/spacenet7_cache/masks
   Average share of 128x128 cells masked in a month pair: 6.5%

Part 1 complete!


## Step 14: Create the train / validation / test split

The locations are split **once, here**, and every later notebook reads the same file:
- **Part 2** fits its PCA feature compression on the **training** locations only.
- **Part 3** trains, selects, and tests on these exact locations.

| Split | Share | Locations (of 60) | Used for |
|---|---|---|---|
| Train | ~70% | 42 | Fitting PCA (Part 2) and training the model (Part 3) |
| Validation | ~15% | 9 | Picking the best epoch, early stopping, and the decision threshold |
| Test | ~15% | 9 | Final evaluation, used once |

The split is **by location**, never by month: months of the same place are strongly correlated, so mixing them across splits would leak answers.

It's saved to `training/split.json`. If that file already exists, it's **kept** after checking that it lists exactly the locations that have labels. That way results from different runs stay comparable. The shuffle uses a fixed seed (42), so the same split is created every time.

In [ ]:
import random

SPLIT_SEED, VAL_FRACTION, TEST_FRACTION = 42, 0.15, 0.15
split_path = Path(config.cache_dir) / "training" / "split.json"
location_ids = sorted(f.name.removesuffix("_labels.npz") for f in drive_labels.glob("*_labels.npz"))

if split_path.exists():
    split = json.loads(split_path.read_text())
    if sorted(sum(split.values(), [])) != location_ids:
        raise ValueError(f"{split_path} lists different locations than have labels. Delete it to create a new split.")
    print(f"Keeping existing split: {split_path}")
else:
    shuffled = location_ids.copy()
    random.Random(SPLIT_SEED).shuffle(shuffled)
    n_test = max(1, round(len(shuffled) * TEST_FRACTION))
    n_val = max(1, round(len(shuffled) * VAL_FRACTION))
    split = {
        "train": sorted(shuffled[n_test + n_val:]),
        "val": sorted(shuffled[n_test:n_test + n_val]),
        "test": sorted(shuffled[:n_test]),
    }
    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_path.write_text(json.dumps(split, indent=2))
    print(f"Created split: {split_path}")

for name, ids in split.items():
    print(f"   {name:5s}: {len(ids)} locations")
print("\nPart 1 complete!")